#Install Libraries

In [2]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q langchain-huggingface
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q sentence-transformers

# Upload PDF

In [3]:
from google.colab import files

uploaded = files.upload()

Saving islreport.pdf to islreport (1).pdf


In [2]:
import os

print(os.listdir())

['.config', 'islreport.pdf', 'sample_data']


# Load PDF

In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_file = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_file)
documents = loader.load()

print("Pages Loaded:", len(documents))

/tmp/ipykernel_15606/3338866086.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages Loaded: 36


# View Sample Text

In [6]:
print(documents[0].page_content[:1000])

IN-HOUSE PRACTICAL TRAINING 
On 
Indian Sign Language Recognition 
System using computer vision and 
Deep Learning 
Submitted to: 
Amity University Noida 
 
In partial fulfillment of the requirements for the award of the degrees 
Of 
Bachelor of Technology 
in 
                                              Computer Science and Engineering 
By 
              Siddhi Sahu                                                        Simran 
                    (A2305223365)                                                           (A2305223366) 
Under the guidance of 
Dr . Smriti Sehgal 
Department of Computer Science and Engineering 
Amity School of Engineering and Technology 
Amity University, Uttar Pradesh


# Split Into Chunks

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print("Total Chunks:", len(docs))

Total Chunks: 149


# Create Embeddings

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Create FAISS Vector Store

In [9]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    docs,
    embeddings
)

print("FAISS Vector Store Created Successfully")

FAISS Vector Store Created Successfully


# Gemini API Key

In [10]:
import os

os.environ["GOOGLE_API_KEY"] = "Use your own api key here"

# Load Gemini

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("Gemini Loaded")

Gemini Loaded


#RAG Function

In [15]:
def ask_question(question):

    retriever = vector_store.as_retriever(
        search_kwargs={"k": 3}
    )

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    prompt = f"""
Answer ONLY from the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    print("=" * 70)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(response.content)

    print("\nRETRIEVED CHUNKS:")
    print("=" * 70)

    for i, doc in enumerate(retrieved_docs):
        print(f"\nChunk {i+1}")
        print(doc.page_content[:300])

#Test

In [16]:
ask_question("What is this document all about?")

QUESTION:
What is this document all about?

ANSWER:
This document is an original piece of work in science and engineering. It appears to be an academic report or similar document, as indicated by the mention of "academic composing," "report," and its table of contents including sections like "DECLARATION," "CERTIFICATE," and "ACKNOWLEDGEMENT." It aims to advance society and aligns with the NTCC framework's goals.

RETRIEVED CHUNKS:

Chunk 1
science and engineering and is an original piece of work. 
The author affirms that consent has been obtained for the utilization of any copyrighted material 
showing up in the report, other than brief passages requiring as it were legitimate affirmation 
in academic composing is acknowledged. 
Date:

Chunk 2
TABLE OF CONTENT 
TITLES                                                                                            PAGE NO. 
DECLARATION …………………………..………………………………………................I                                                               

In [17]:
ask_question("What model architecture was used?")

QUESTION:
What model architecture was used?

ANSWER:
Convolutional Neural Networks (CNNs)

RETRIEVED CHUNKS:

Chunk 1
determining the overall model’s performance.

Chunk 2
23 
  
All the techniques that we used to train the model, including MediaPipes’s hand detection, data 
augmentation, preprocessing, early stopping all those helped model to learn effectively.  
So after we train the model over 15 epochs, we generate a classification report and this 
classification 

Chunk 3
implemented, which consists of grayscale conversion, normalization, resizing , and data 
augmentation to refine the model ’s versatility and performance. Convolutional Neural 
Networks (CNNs), which is also known for their robustness in object classification in images, 
were used for model implement


In [18]:
ask_question("What accuracy was achieved?")

QUESTION:
What accuracy was achieved?

ANSWER:
Training accuracy of 92.38%, validation accuracy of 97.37%, and a prediction accuracy of 60.8% on real-world images were achieved.

RETRIEVED CHUNKS:

Chunk 1
23 
  
All the techniques that we used to train the model, including MediaPipes’s hand detection, data 
augmentation, preprocessing, early stopping all those helped model to learn effectively.  
So after we train the model over 15 epochs, we generate a classification report and this 
classification 

Chunk 2
model secured high performance, with a training accuracy of 92.38% and validation accuracy 
of 97.37%, with a weighted F1-score of 0.98, showing its ability to accurately classify hand. 
However, testing on 46 world images gave a prediction accuracy of 60.8% showing the huge 
gap between the validat

Chunk 3
25 
  
Though high validation and training accuracy is shown and steady rise in the curves over the 
time were shown, however when tested on real world images, models perform

#Interactive Chat

In [19]:
while True:

    question = input("\nAsk Question (type exit to stop): ")

    if question.lower() == "exit":
        break

    ask_question(question)


Ask Question (type exit to stop): what are the names of student who made this report?
QUESTION:
what are the names of student who made this report?

ANSWER:
Siddhi Sahu and Simran

RETRIEVED CHUNKS:

Chunk 1
science and engineering and is an original piece of work. 
The author affirms that consent has been obtained for the utilization of any copyrighted material 
showing up in the report, other than brief passages requiring as it were legitimate affirmation 
in academic composing is acknowledged. 
Date:

Chunk 2
ACKNOWLEDGEMENT 
 
We would like to express our gratitude and appreciation to everyone who helped us in 
completing this report and also for supporting us throughout the development of this paper. 
Special thanks to our faculty guide Dr. Smriti Sehgal for providing her valuable suggestions 
and supp

Chunk 3
23 
  
All the techniques that we used to train the model, including MediaPipes’s hand detection, data 
augmentation, preprocessing, early stopping all those helped model 